# Bayesian A/B testing, worked by hand then by AB Advisor

This notebook is the math behind the Streamlit app: why Beta–Binomial posteriors exist, how P(better) is estimated, and how expected loss becomes a ship/hold rule.

## 1. The question a p-value does not answer

A two-proportion z-test asks: *if both variants had the same conversion rate, how often would we see a difference this large?* That probability is **not** P(the new checkout is better).

Bayesian analysis starts with a prior on each variant's conversion rate $\theta$, updates with binomial data, and reads probabilities off the posterior.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.bayesian import AnalysisConfig, MetricType, analyze_metric, highest_density_interval
from src.data_gen import generate_checkout_experiment
from src.decisions import decide_experiment
from src.metrics import load_experiment
from src.summarize import generate_insights

## 2. Beta–Binomial in closed form

Prior: $\theta \sim \mathrm{Beta}(\alpha, \beta)$.  
Likelihood: $s$ conversions out of $n$ users.  
Posterior: $\theta \mid \text{data} \sim \mathrm{Beta}(\alpha + s,\, \beta + n - s)$.

Default $\alpha = \beta = 1$ is uniform. Historical conversion $p$ with strength $n_0$ is $\mathrm{Beta}(p n_0, (1-p)n_0)$.

In [ ]:
rng = np.random.default_rng(0)
control = rng.binomial(1, 0.10, size=5000)
treatment = rng.binomial(1, 0.125, size=5000)

alpha, beta = 1.0, 1.0
post_c = (alpha + control.sum(), beta + len(control) - control.sum())
post_t = (alpha + treatment.sum(), beta + len(treatment) - treatment.sum())
print("control Beta", post_c, "mean", post_c[0] / sum(post_c))
print("treatment Beta", post_t, "mean", post_t[0] / sum(post_t))

c_s = rng.beta(*post_c, size=20_000)
t_s = rng.beta(*post_t, size=20_000)
print("P(treatment > control)", np.mean(t_s > c_s))
lift = (t_s - c_s) / c_s
print("expected lift", lift.mean(), "90% HDI", highest_density_interval(lift, 0.9))

## 3. Expected loss

Loss of shipping treatment = average amount conversion is worse *when it is worse*:
$\mathbb{E}[\max(\theta_C - \theta_T, 0)]$.

Ship when that number is below a product threshold (for example 0.001 conversion points), not merely when P(better) is large. A 97% chance of a +0.1% lift may not be worth the engineering cost.

In [ ]:
loss_ship = np.mean(np.maximum(c_s - t_s, 0))
loss_hold = np.mean(np.maximum(t_s - c_s, 0))
print("E[loss] ship", loss_ship, "hold", loss_hold)

## 4. Run the full sample experiment through AB Advisor

In [ ]:
df = generate_checkout_experiment(n_per_arm=5000, seed=42)
data = load_experiment(df)
print(data.n_control, data.n_treatment, "SRM p", round(data.srm_pvalue, 3))
print(data.types)

cfg = AnalysisConfig(n_samples=12_000, seed=42)
results = [
    analyze_metric(*data.split(name), name, data.types[name], cfg)
    for name in data.metric_cols
]
for r in results:
    print(
        f"{r.metric_name:20s}  P(better)={r.prob_improvement:.1%}  "
        f"E[lift]={r.expected_relative_lift:+.1%}  "
        f"HDI=[{r.relative_lift_hdi[0]:+.1%}, {r.relative_lift_hdi[1]:+.1%}]"
    )

decision = decide_experiment(
    results,
    srm_flag=data.srm_flag,
    higher_is_better={"bounce": False},
)
print(decision.headline)
md, src, _ = generate_insights(results, decision, use_llm=False)
print(src)
print(md[:800])

## 5. Other likelihoods

- **Normal** with unknown variance: posterior of the mean is Student-t.
- **Log-normal**: the same t / inverse-χ² update on $\log y$, then $\mathbb{E}[Y] = \exp(\mu + \sigma^2/2)$.
- **Poisson**: $\lambda \sim \mathrm{Gamma}(\alpha + \sum x, \beta + n)$.
- **Hurdle revenue**: zeros are non-converters; positives are log-normal spend. ARPU $= p \times \mathbb{E}[y \mid y>0]$.

These are implemented in `src/bayesian.py`. MCMC (PyMC) is unnecessary for this conjugate set and would make a Streamlit dashboard slow.

## 6. Caveats to repeat in every review

Independent users, random assignment, a model that matches the metric, a pre-registered primary KPI, and a pre-registered ship threshold. Peeking is allowed for these Bayesian rules and forbidden for the p-value footnote.